# General overview
Log transofrming and vice versa yielded a very low squared, so we need to do sensitivity analysis to check for further discrepancies in the data.


1. Load data, define features/target (same as before)

2. Set up RepeatedKFold — this is the tool that creates multiple train/test splits automatically

3. For each split, build and train two versions of each model: original-scale and log-transformed

4. Record R² for both versions, every split

5. Average the results and compare

## Step 1: Load Data/ Define Features and target


In [25]:
import pandas as pd
import numpy as np
from sklearn.model_selection import RepeatedKFold
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import r2_score

RANDOM_STATE = 42

# Load data fresh
df = pd.read_csv("final_extract_dataset.csv")

feature_cols = [
    'Region_Code', 'Median_GDP', 'Median_Population_Density',
    'Median_Urban_pct', 'Median_Sanitation_Access_pct',
    'Median_Rainfall', 'Median_Temp'
]

X = df[feature_cols]
y = df['Median_Malaria_Cases']
#y = np.log1p(df['Median_Malaria_Cases'])

## Step 2: set up repeated kfold

In [26]:
# This creates the "repeated splitting" tool
rkf = RepeatedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)

print("Total number of train/test cycles:", rkf.get_n_splits())
#Ms Maureen said we needed the datat to be tested with different configurations of train 
# and test data. this yields 25 different train test experiments to see if the r^2 really is
# low across all or just one split we did in notebook 2 

Total number of train/test cycles: 25


# Step 3 build and train 2 versions per model

In [27]:
# --- Preprocessing setup (reused inside every fold) ---
categorical_cols = ['Region_Code']
numeric_cols = [
    'Median_GDP', 'Median_Population_Density', 'Median_Urban_pct',
    'Median_Sanitation_Access_pct', 'Median_Rainfall', 'Median_Temp'
]

def build_preprocessor():
    return ColumnTransformer(transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numeric_cols)
    ])

"""
this is wrapped in a function (build_preprocessor()) instead of being one 
fixed object like before. Why? Because we're about to train models 25 times over, on 25 different train/test splits — each fold needs its own fresh, independently-fitted preprocessor, so nothing "leaks" information between folds. Calling this function each time gives us a brand-new, untrained ColumnTransformer for every fold.
"""

'\nthis is wrapped in a function (build_preprocessor()) instead of being one \nfixed object like before. Why? Because we\'re about to train models 25 times over, on 25 different train/test splits — each fold needs its own fresh, independently-fitted preprocessor, so nothing "leaks" information between folds. Calling this function each time gives us a brand-new, untrained ColumnTransformer for every fold.\n'

In [28]:
alphas_wide = np.logspace(-3, 6, 50)  # slightly fewer alphas than before, since this runs 25x now
#I reduced the alpha search from 100 to 50 candidates (np.logspace(-3, 6, 50) instead of 100) purely for speed — with 25 folds × 8 models × alpha search each, this could get slow otherwise. 50 candidates is still plenty for a reliable search.

def build_models():
    """Returns a fresh dictionary of un-trained models, both original and log-transformed versions."""
    
    original_models = {
        'MLR': LinearRegression(),
        'Ridge': RidgeCV(alphas=alphas_wide, cv=5),
        'Lasso': LassoCV(alphas=alphas_wide, cv=5, random_state=RANDOM_STATE, max_iter=10000),
        'ElasticNet': ElasticNetCV(alphas=alphas_wide, cv=5, random_state=RANDOM_STATE, max_iter=10000),
    }
    
    log_models = {
        'MLR_log': TransformedTargetRegressor(LinearRegression(), func=np.log1p, inverse_func=np.expm1),
        'Ridge_log': TransformedTargetRegressor(RidgeCV(alphas=alphas_wide, cv=5), func=np.log1p, inverse_func=np.expm1),
        'Lasso_log': TransformedTargetRegressor(LassoCV(alphas=alphas_wide, cv=5, random_state=RANDOM_STATE, max_iter=10000), func=np.log1p, inverse_func=np.expm1),
        'ElasticNet_log': TransformedTargetRegressor(ElasticNetCV(alphas=alphas_wide, cv=5, random_state=RANDOM_STATE, max_iter=10000), func=np.log1p, inverse_func=np.expm1),
    }
    
    return {**original_models, **log_models}

## Step 4: Record all the R^2 for all models and their log twins

In [29]:
from sklearn.model_selection import KFold

kf_single = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

single_fold_results = []
fold_number = 0

for train_idx, test_idx in kf_single.split(X):
    fold_number += 1

    X_train_fold = X.iloc[train_idx]
    X_test_fold = X.iloc[test_idx]
    y_train_fold = y.iloc[train_idx]
    y_test_fold = y.iloc[test_idx]

    models = build_models()

    for model_name, model in models.items():
        preprocessor = build_preprocessor()
        pipe = Pipeline([
            ('preprocessor', preprocessor),
            ('regressor', model)
        ])

        pipe.fit(X_train_fold, y_train_fold)

        # Raw-scale R² (works for ALL models, log or not)
        preds_raw = pipe.predict(X_test_fold)
        r2_raw = r2_score(y_test_fold, preds_raw)

        # Log-scale R² (only meaningful for _log models)
        r2_log = np.nan
        if model_name.endswith('_log'):
            X_test_processed = pipe.named_steps['preprocessor'].transform(X_test_fold)
            preds_log_scale = pipe.named_steps['regressor'].regressor_.predict(X_test_processed)
            y_test_fold_log = np.log1p(y_test_fold)
            r2_log = r2_score(y_test_fold_log, preds_log_scale)

        single_fold_results.append({
            'fold': fold_number,
            'model': model_name,
            'r2_raw': r2_raw,
            'r2_log': r2_log
        })

print(f"Done. Total results recorded: {len(single_fold_results)}")

Done. Total results recorded: 40


## Step 5: Average results for comparison

In [33]:
single_df = pd.DataFrame(single_fold_results)

# One row per model, raw R² mean/std next to log R² mean/std
side_by_side = single_df.groupby('model').agg(
    Mean_R2_Raw=('r2_raw', 'mean'),
    Std_R2_Raw=('r2_raw', 'std'),
    Mean_R2_Log=('r2_log', 'mean'),
    Std_R2_Log=('r2_log', 'std')
).round(4)

side_by_side = side_by_side.sort_values('Mean_R2_Raw', ascending=False)
side_by_side



,Mean_R2_Raw,Std_R2_Raw,Mean_R2_Log,Std_R2_Log
model,,,,
Ridge_log,0.0027,0.1725,0.5291,0.2404
ElasticNet_log,-0.0054,0.1492,0.5298,0.2290
MLR_log,-0.0200,0.1109,0.5108,0.3059
Lasso_log,-0.0540,0.1367,0.4942,0.2911
ElasticNet,-0.3306,1.2551,NaN,NaN
Lasso,-0.3411,1.3369,NaN,NaN
Ridge,-0.4081,1.3458,NaN,NaN
MLR,-0.7047,2.0751,NaN,NaN


In [32]:
# Pivot single_df so each fold's r2_log becomes its own column, per model
log_only = single_df[single_df['model'].str.endswith('_log')]

log_pivot = log_only.pivot(index='model', columns='fold', values='r2_log')
log_pivot.columns = [f'Fold {c}' for c in log_pivot.columns]

log_pivot['Mean R²'] = log_pivot.mean(axis=1)
log_pivot['Std R²'] = log_pivot.std(axis=1)

# Reorder columns to match your original table's layout
log_pivot = log_pivot[['Mean R²', 'Std R²', 'Fold 1', 'Fold 2', 'Fold 3', 'Fold 4', 'Fold 5']]
log_pivot = log_pivot.sort_values('Mean R²', ascending=False)

log_pivot.round(4)

,Mean R²,Std R²,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5
model,,,,,,,
ElasticNet_log,0.5298,0.2048,0.7107,0.4588,0.7572,0.1831,0.5394
Ridge_log,0.5291,0.2150,0.7007,0.4693,0.7377,0.1417,0.5961
MLR_log,0.5108,0.2736,0.7182,0.4391,0.7627,0.0107,0.6232
Lasso_log,0.4942,0.2603,0.7283,0.4469,0.7209,0.0183,0.5568


This repeated cross-validation is showing us that performance is highly unstable depending on which 20 countries happen to land in our test set. With only 99 countries and likely a handful of extreme outlier countries (very high case counts), a single split can randomly put those outliers all in train (making test look easy) or all in test (making the model look terrible) — and our original split happened to be a relatively favorable one.

A repeated 5×5-fold cross-validation revealed that model performance is highly sensitive to sample composition, with wide variance in R² across folds (std often exceeding the mean). Contrary to the initial single-split evaluation, log-transformed target models consistently outperformed their untransformed counterparts across repeated folds (e.g., Lasso_log mean R²=-0.279 vs Lasso mean R²=-0.477), suggesting the transformation does provide meaningful, consistent benefit despite both remaining below acceptable predictive thresholds. This instability likely stems from the small sample size (n=99) combined with a small number of extreme outlier countries, which disproportionately affect performance depending on their allocation to train or test folds in any given split.